# ChemBreak V10 Cloud
## Standard Google Colab + Vertex AI

This notebook clones your GitHub repository and uses only the `ChemBreak_V10_Cloud` folder.

Start with `test`. Do not run generation until preflight and the fresh assignment-plan review both succeed.


## 1. Python setup


In [ ]:
from pathlib import Path
import subprocess
import sys
import json
import os
import shutil

PROJECT_ID = "rs-foundsecft-mghasemi"

print("Python setup: OK")
print("Project:", PROJECT_ID)


## 2. Authenticate standard Colab to Google Cloud

Open the displayed URL, sign in with the Google account that has access to the project, then paste the verification code back into Colab.


In [ ]:
!gcloud auth application-default login --no-launch-browser


## 3. Attach the project to Application Default Credentials


In [ ]:
!gcloud auth application-default set-quota-project rs-foundsecft-mghasemi
!gcloud config set project rs-foundsecft-mghasemi

import google.auth

credentials, detected_project = google.auth.default(
    scopes=["https://www.googleapis.com/auth/cloud-platform"]
)

print("Authentication: OK")
print("Detected project:", detected_project)
print("Quota project:", credentials.quota_project_id)
print("Credential type:", type(credentials).__name__)


## 4. Mount Google Drive for persistent checkpoints


In [ ]:
USE_GOOGLE_DRIVE = True

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    STORAGE_ROOT = Path("/content/drive/MyDrive/ChemBreak_V10")
else:
    STORAGE_ROOT = Path("/content/ChemBreak_V10")

STORAGE_ROOT.mkdir(parents=True, exist_ok=True)
print("Persistent storage root:", STORAGE_ROOT)


## 5. Clone or refresh the GitHub repository


In [ ]:
REPO_URL = "https://github.com/Jollychuks/ChemBreak.git"
REPO_ROOT = Path("/content/ChemBreak_repo")
PROJECT_SUBDIR = "ChemBreak_V10_Cloud"

if (REPO_ROOT / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_ROOT), "pull", "--ff-only"], check=True)
elif REPO_ROOT.exists():
    shutil.rmtree(REPO_ROOT)
    subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)

PROJECT_DIR = REPO_ROOT / PROJECT_SUBDIR
if not PROJECT_DIR.is_dir():
    raise FileNotFoundError(
        f"{PROJECT_SUBDIR} was not found in GitHub. "
        "Upload the complete V10 folder to the repository first."
    )

PIPELINE = PROJECT_DIR / "scripts" / "chembreak_v10_cloud.py"
CONFIG_SOURCE = PROJECT_DIR / "config" / "run_config.json"

print("V10 folder:", PROJECT_DIR)
print("Pipeline:", PIPELINE)


## 6. Install V10 requirements


In [ ]:
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_DIR / "requirements.txt")],
    check=True
)
print("V10 requirements installed.")


## 7. Choose the run

Keep `RUN_TYPE = "test"` until the full 9-task workflow has been reviewed.


In [ ]:
RUN_TYPE = "test"  # "test", "pilot", or "production"
GCS_OUTPUT_URI = ""

RUNTIME_DIR = Path("/content/ChemBreak_V10_runtime")
RUNTIME_DIR.mkdir(parents=True, exist_ok=True)
RUNTIME_CONFIG = RUNTIME_DIR / f"run_config_{RUN_TYPE}.json"

cfg = json.loads(CONFIG_SOURCE.read_text(encoding="utf-8"))
cfg["run_type"] = RUN_TYPE
cfg["project_id"] = PROJECT_ID
cfg["gcs_output_uri"] = GCS_OUTPUT_URI
RUNTIME_CONFIG.write_text(json.dumps(cfg, indent=2), encoding="utf-8")

OUTPUT_DIR = STORAGE_ROOT / "outputs" / RUN_TYPE
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def run_stage(stage):
    import time
    command = [
        sys.executable, "-u", str(PIPELINE),
        "--stage", stage,
        "--project-dir", str(PROJECT_DIR),
        "--config", str(RUNTIME_CONFIG),
        "--output-dir", str(OUTPUT_DIR),
    ]
    started = time.time()
    print(f"\n===== V10 {stage.upper()} START =====", flush=True)
    print(f"Output directory: {OUTPUT_DIR}", flush=True)
    subprocess.run(command, check=True)
    elapsed = time.time() - started
    print(
        f"===== V10 {stage.upper()} DONE | "
        f"elapsed {elapsed/60:.1f} min =====\n",
        flush=True,
    )

print("Run type:", RUN_TYPE)
print("Output directory:", OUTPUT_DIR)


## 8. Preflight Vertex AI model access

The required endpoint roles must all report `OK`. This cell stops the notebook if a required role fails.


In [ ]:
run_stage("preflight")

import pandas as pd
from IPython.display import display

preflight = pd.read_csv(OUTPUT_DIR / "preflight_models.csv")
display(preflight)

cfg_now = json.loads(RUNTIME_CONFIG.read_text(encoding="utf-8"))
required_roles = set(cfg_now["generator_roles"] + cfg_now["judge_roles"] + ["repair_model", "adjudicator"])
required_rows = preflight[preflight["role"].isin(required_roles)]
bad = required_rows[required_rows["status"] != "OK"]

if not bad.empty:
    raise RuntimeError(
        "V10 preflight failed for required roles:\n" +
        bad[["role", "model", "status", "detail"]].to_string(index=False)
    )

print("All required V10 model roles passed preflight.")


## 9. Bootstrap fresh V10 source provenance


In [ ]:
run_stage("bootstrap")


## 10. Build the fresh V10 assignment plan


In [ ]:
run_stage("plan")


## 11. Inspect V10 assignment coverage before generation

Review this output before spending generation calls.


In [ ]:
plan = pd.read_csv(OUTPUT_DIR / "assignments_v10.csv")

display(plan.head(25))
print("Assignments:", len(plan))

print("\nHC coverage")
display(plan["hc_id"].value_counts().sort_index())

print("\nHD coverage")
display(plan["hd_id"].value_counts().sort_index())

print("\nOT coverage")
display(plan["ot_id"].value_counts().sort_index())

print("\nRequest-form coverage")
display(plan["request_form"].value_counts())


## 12. Generate the candidate pool

V10 prints live progress for every model call. While a Vertex request is still running, a heartbeat appears about every 20 seconds. After each candidate it prints the completed count, percentage, elapsed time, and estimated time remaining.


In [ ]:
run_stage("generate")


## 13. Deterministic validation

Validation prints every candidate as it is checked, including PASS/FAIL, defect preview, percentage, elapsed time, and ETA.


In [ ]:
run_stage("validate")


## 14. Repair invalid candidates

Repair prints each repair request before it is sent, emits a waiting heartbeat during long Vertex calls, and reports PASS/FAIL plus overall progress and ETA.


In [ ]:
run_stage("repair")


## 15. Blind judging and adjudication

Judging shows the current assignment, candidate pair, each judge call, each judge decision, disagreements, adjudication activity, selected candidate, percentage, elapsed time, and ETA.


In [ ]:
run_stage("judge")


## 16. Refill unresolved assignments, then judge again

Refill has the same live heartbeat, per-assignment status, elapsed time, and ETA. The second judging pass is also fully visible.


In [ ]:
run_stage("refill")
run_stage("judge")


## 17. Finalize and inspect the task bank


In [ ]:
run_stage("finalize")
run_stage("status")

for name in [
    "run_summary.json",
    "coverage_report.csv",
    "diversity_report.csv",
    "final_task_bank.csv",
]:
    path = OUTPUT_DIR / name
    print("\n", name)
    if path.suffix == ".json" and path.exists():
        print(path.read_text(encoding="utf-8"))
    elif path.exists():
        display(pd.read_csv(path).head(30))
    else:
        print("not written")


## 18. Create a persistent checkpoint ZIP


In [ ]:
summary = json.loads((OUTPUT_DIR / "run_summary.json").read_text(encoding="utf-8"))
label = summary["completion_label"]

archive = shutil.make_archive(
    str(STORAGE_ROOT / f"ChemBreak_V10_{RUN_TYPE}_{label}"),
    "zip",
    root_dir=str(OUTPUT_DIR)
)

print("Checkpoint ZIP:", archive)
